In [ ]:
import os, shutil, datetime, itertools
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
from pathlib import Path
from google.colab import drive
import keras_tuner as kt

In [ ]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.6 MB/s eta 0:00:00


In [ ]:

# ---------------- Mount & Paths ----------------
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split80_20"  # train & val
TEST_DIR  = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DRIVE_ROOT}/DL/Emotion_EfficientNet_NaiveBayes_Tuned_80_20_with pt{ts}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

fresh_copy(os.path.join(DATA_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(DATA_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)


Mounted at /content/drive


In [ ]:
# ---------------- Data pipeline ----------------
IMG_SIZE_RAW = (48, 48)    # dataset images (grayscale)
IMG_SIZE_INP = (224, 224)  # model input
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def make_loader(dir_path, shuffle=True, batch=BATCH):
    ds = image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="categorical",   # one-hot for Keras metrics/ROC
        color_mode="grayscale",
        image_size=IMG_SIZE_RAW,
        batch_size=batch,
        shuffle=shuffle,
        seed=SEED
    )
    class_names = ds.class_names

    def _prep(x, y):
        x = tf.image.grayscale_to_rgb(x)
        x = tf.image.resize(x, IMG_SIZE_INP)
        x = tf.cast(x, tf.float32) / 255.0  # keep [0,1]; branch-wise preprocess later
        return x, y

    return ds.map(_prep, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), class_names

train_ds, class_names = make_loader(LOCAL_TRAIN, shuffle=True)
val_ds, _             = make_loader(LOCAL_VAL,   shuffle=False)
test_ds, _            = make_loader(LOCAL_TEST,  shuffle=False)
num_classes = len(class_names)
print("Classes:", class_names, "| Num classes:", num_classes)

Found 22965 files belonging to 7 classes.
Found 5744 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'] | Num classes: 7


In [ ]:

# ---------------- EfficientNet Feature Extraction ----------------
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_pp

# EfficientNetB0 model (pre-trained)
efficientnet_base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
efficientnet_base.trainable = False

inp = layers.Input(shape=(224,224,3), name="inp_rgb01")  # [0,1] RGB

# EfficientNet preprocess & features
x = layers.Lambda(lambda z: efficientnet_pp(z*255.0), name="efficientnet_preproc")(inp)
f = efficientnet_base(x, training=False)
f = layers.GlobalAveragePooling2D(name="gap_efficientnet")(f)

# Classifier head (Naive Bayes)
out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

model = models.Model(inp, out, name="EfficientNet_NaiveBayes_Hybrid")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "EfficientNet_NaiveBayes_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inp_rgb01 (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnet_preproc (Lambda)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap_efficientnet                │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,058,538 (15.48 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# ---------------- Hyperparameter Tuning Function ----------------

def build_model(hp):
    efficientnet_base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(224,224,3))
    efficientnet_base.trainable = False

    inp = layers.Input(shape=(224,224,3), name="inp_rgb01")

    # EfficientNet preprocess & features
    x = layers.Lambda(lambda z: efficientnet_pp(z*255.0), name="efficientnet_preproc")(inp)
    f = efficientnet_base(x, training=False)
    f = layers.GlobalAveragePooling2D(name="gap_efficientnet")(f)

    # Hyperparameter tuning for the Dense layer and Dropout
    f = layers.Dense(
        hp.Int('dense_units', min_value=256, max_value=1024, step=128),
        activation="relu"
    )(f)

    f = layers.BatchNormalization()(f)
    f = layers.Dropout(hp.Float('dropout_rate', min_value=0.3, max_value=0.7, step=0.1))(f)

    out = layers.Dense(num_classes, activation="softmax", name="pred")(f)

    model = models.Model(inp, out, name="EfficientNet_NaiveBayes_Hybrid")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-3, sampling='LOG')),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ---------------- Keras Tuner ----------------
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=2,  # More epochs for better tuning
    hyperband_iterations=2,  # Number of iterations for tuning
    directory=OUT_DIR,
    project_name="efficientnet_naivebayes_tuning"
)


In [ ]:

# ---------------- Tuning the Model ----------------
tuner.search(train_ds, validation_data=val_ds, epochs=2, batch_size=BATCH)

# Get the best model
best_model = tuner.get_best_models()[0]
best_model.summary()

Trial 4 Complete [00h 02m 00s]
val_accuracy: 0.5179317593574524

Best val_accuracy So Far: 0.5252437591552734
Total elapsed time: 00h 08m 20s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "EfficientNet_NaiveBayes_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inp_rgb01 (InputLayer)          │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnet_preproc (Lambda)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap_efficientnet                │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 896)            │     1,147,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 896)            │         3,584 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 896)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pred (Dense)                    │ (None, 7)              │         6,279 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,207,210 (19.86 MB)

 Trainable params: 1,155,847 (4.41 MB)

 Non-trainable params: 4,051,363 (15.45 MB)

In [ ]:
# ---------------- Save the Best Model ----------------
best_model.save(os.path.join(OUT_DIR, "efficientnet_naivebayes_best_tuned.keras"))

# ---------------- Training the Best Model ----------------
history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,  # After tuning, train for more epochs
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR, "best_model_tuned.keras"), save_best_only=True),
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)

# Save the final model and training history
best_model.save(os.path.join(OUT_DIR, "efficientnet_naivebayes_final_tuned.keras"))
pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, "training_history_tuned.csv"), index=False)


Epoch 1/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 81s 83ms/step - accuracy: 0.4673 - loss: 1.4868 - val_accuracy: 0.5306 - val_loss: 1.2946
Epoch 2/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.4892 - loss: 1.3889 - val_accuracy: 0.5317 - val_loss: 1.2537
Epoch 3/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.5107 - loss: 1.3246 - val_accuracy: 0.5406 - val_loss: 1.2394
Epoch 4/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.5254 - loss: 1.2927 - val_accuracy: 0.5487 - val_loss: 1.2207
Epoch 5/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 37s 52ms/step - accuracy: 0.5200 - loss: 1.2695 - val_accuracy: 0.5503 - val_loss: 1.2114
Epoch 6/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.5321 - loss: 1.2499 - val_accuracy: 0.5498 - val_loss: 1.2099
Epoch 7/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.5476 - loss: 1.2221 - val_accuracy: 0.5521 - val_loss: 1.2029
Epoch 8/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.5499 - loss: 1.2149 - 

In [ ]:
# ---------------- Curves ----------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.title("Accuracy Curve"); plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "training_curves_tuned.png"), dpi=150)
plt.close()
print("Training curves saved.")

# ---------------- Evaluate on Test ----------------
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"TEST -> loss: {test_loss:.4f} | acc: {test_acc:.4f}")

# ---------------- Predictions & Reports ----------------
y_true = []
for _, y in test_ds.unbatch():
    y_true.append(y.numpy())
y_true = np.array(y_true)
y_true_cls = np.argmax(y_true, axis=1)

y_prob = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# Classification Report
rep = classification_report(y_true_cls, y_pred, target_names=class_names, digits=4)
print(rep)
with open(os.path.join(OUT_DIR, "classification_report_tuned.txt"), "w") as f:
    f.write(rep)

# Confusion Matrix
cm = confusion_matrix(y_true_cls, y_pred)
plt.figure(figsize=(7.2,6.2))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("Confusion Matrix"); plt.colorbar()
ticks = np.arange(num_classes)
plt.xticks(ticks, class_names, rotation=45)
plt.yticks(ticks, class_names)
plt.ylim(num_classes-0.5, -0.5) # Corrected y-axis limits
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix_tuned.png"), dpi=150)
plt.close()
print("Confusion matrix saved.")

# ROC Curve (example for one class)
# For a multi-class problem, you might want to plot ROC for each class or micro/macro average
# This requires converting y_true_cls to one-hot or using OneVsRestClassifier approach with scikit-learn
# Here's an example for one class (e.g., 'happy')
try:
    happy_class_index = class_names.index('happy')
    fpr, tpr, _ = roc_curve(y_true[:, happy_class_index], y_prob[:, happy_class_index])
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for class: happy')
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(OUT_DIR, "roc_curve_happy_tuned.png"), dpi=150)
    plt.close()
    print("ROC curve for 'happy' class saved.")
except ValueError:
    print("Class 'happy' not found in class_names. Skipping ROC curve generation.")

Training curves saved.
TEST -> loss: 1.2073 | acc: 0.5549
              precision    recall  f1-score   support

       angry     0.4563    0.4363    0.4461       958
   disgusted     0.5294    0.3243    0.4022       111
     fearful     0.4790    0.2344    0.3148      1024
       happy     0.6740    0.8298    0.7438      1774
     neutral     0.5266    0.4899    0.5076      1233
         sad     0.4174    0.5269    0.4658      1247
   surprised     0.7056    0.6691    0.6868       831

    accuracy                         0.5549      7178
   macro avg     0.5412    0.5015    0.5096      7178
weighted avg     0.5487    0.5549    0.5421      7178

Confusion matrix saved.
ROC curve for 'happy' class saved.
